# Targeted Theme Analysis — Deep Colonial Imagery Detection

**Goal:** Take a single *theme* (e.g. "beach", "life in Jaffa", "market scene") and generate a large number of variations (~100) for each identity framing. With enough seeds, individual noise cancels out and **systematic visual differences** emerge with high statistical confidence.

## Identity Vector Method

Instead of using different text prompts (which introduces text-encoder variability as a confound), we compute **identity direction vectors** in prompt-embedding space:

$$\vec{v}_\text{arab} = \text{encode}(\text{"Arab person"}) - \text{encode}(\text{"a person"})$$

Then for any theme prompt, the conditioned embedding is:

$$\vec{e}_\text{conditioned} = \text{encode}(\text{theme}) + \alpha \cdot \vec{v}_\text{identity}$$

This gives **perfect experimental control**: the ONLY difference between categories is the identity vector. No text encoder variability, no tokenization artifacts, no contextual interaction between theme words and identity words.

---
### Workflow
1. **Define theme** — a single scene description  
2. **Compute identity vectors** — encode anchor + identity phrases, take differences
3. **Generate 100+ images** per identity (neutral + each identity vector)
4. **Statistical analysis** — divergence, flow, spatial, motif extraction  
5. **Differential heatmaps** — where exactly in the image does identity change things?  
6. **Image evolution** — start from a reference image, vary denoising strength with identity vectors

In [1]:
import sys
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from io import BytesIO
import json
import time
from collections import defaultdict

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from client.interface import SlopClient, InferenceResult
from client.config import registry
from client.analysis.colonial import (
    ColonialGrammarAnalysis,
    # Identity vector infrastructure
    IdentityVectorSet,
    compute_identity_vectors,
    build_conditioned_embeds,
    sweep_identity_strength,
    # Batch-aware statistics
    batch_divergence,
    batch_flow_difference,
    batch_channel_difference,
    batch_spatial_heatmap,
    batch_inter_category_pca,
    BatchDivergenceResult,
    BatchFlowResult,
    BatchChannelResult,
    BatchSpatialResult,
    # Divergence rate analysis
    batch_divergence_rate,
    compare_divergence_rates,
    DivergenceRateResult,
    # Spatial motif extraction
    extract_motif_direction,
    measure_motif_activation,
    extract_motif_components,
    MotifDirection,
    MotifExtractionResult,
    # Visualization helpers
    average_trajectories,
    compute_flow_field,
    average_flow_field,
    flow_difference,
    flow_magnitude,
    spatial_difference_heatmap,
    temporal_heatmap_sequence,
)

print("Imports OK ✓")

/Users/pkd/Desktop/slop/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK ✓


## Configuration

Change `THEME` to target any scene. `IDENTITIES` defines the direction vectors that will be computed in embedding space. The `ANCHOR` phrase is subtracted to isolate just the identity direction.

In [2]:
# ═══════════════════════════════════════════════════════════════════════
#  EXPERIMENT PARAMETERS
# ═══════════════════════════════════════════════════════════════════════

THEME = "daily life in Jaffa"

# Identity vector definitions
# Each is a phrase whose embedding minus the ANCHOR gives the direction vector
ANCHOR = "a person"
IDENTITIES = {
    "arab":        "Arab person",
    "palestinian": "Palestinian person",
    "jewish":      "Jewish person",
    "jesus":       "person in the time of Jesus",
}

# Identity vector scale (1.0 = full strength, 0.5 = half, etc.)
IDENTITY_SCALE = 1.0

# ── Server ──
SERVER_NAME = "vast-auto-test"
MODEL = "runwayml/stable-diffusion-v1-5"

# ── Generation params ──
N_SEEDS = 100              # seeds per identity — the power of this notebook
NUM_STEPS = 50
GUIDANCE_SCALE = 7.5
SEED_START = 0

# ── Batching (to avoid overloading the server) ──
BATCH_SIZE = 10            # generate this many, save, continue

# ═══════════════════════════════════════════════════════════════════════

# Categories: neutral (no vector) + all identities
CATEGORIES = ["neutral"] + list(IDENTITIES.keys())

# Output directory
OUTPUT_DIR = project_root / "results" / f"targeted_{THEME.replace(' ', '_').lower()}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Color palette for consistent plotting
COLORS = {
    "neutral":     "#607D8B",
    "arab":        "#FF5722",
    "palestinian": "#4CAF50",
    "jewish":      "#2196F3",
    "jesus":       "#9C27B0",
}

cfg = registry.get(SERVER_NAME)
if not cfg:
    raise ValueError(f"Server '{SERVER_NAME}' not found. Run deploy first.")

print(f"Theme:         \"{THEME}\"")
print(f"Anchor:        \"{ANCHOR}\"")
print(f"Scale:         {IDENTITY_SCALE}")
print(f"Server:        {SERVER_NAME}")
print(f"Model:         {MODEL}")
print(f"Seeds/cat:     {N_SEEDS}")
print(f"Steps:         {NUM_STEPS}")
print(f"Output:        {OUTPUT_DIR}")
print(f"\nIdentity vectors to compute:")
for name, phrase in IDENTITIES.items():
    print(f"  {name:>13}: encode(\"{phrase}\") | encode(\"{ANCHOR}\")")
print(f"\nCategories: {CATEGORIES}")

Theme:         "daily life in Jaffa"
Anchor:        "a person"
Scale:         1.0
Server:        vast-auto-test
Model:         runwayml/stable-diffusion-v1-5
Seeds/cat:     100
Steps:         50
Output:        /Users/pkd/Desktop/slop/results/targeted_daily_life_in_jaffa

Identity vectors to compute:
           arab: encode("Arab person") | encode("a person")
    palestinian: encode("Palestinian person") | encode("a person")
         jewish: encode("Jewish person") | encode("a person")
          jesus: encode("person in the time of Jesus") | encode("a person")

Categories: ['neutral', 'arab', 'palestinian', 'jewish', 'jesus']


### Compute Identity Vectors

Connect to the server, encode all phrases through the model's text encoder, and compute direction vectors. Also encodes the theme prompt. These are cached to disk so you don't need to recompute on re-runs.

In [3]:
# ── Compute identity vectors (or load from cache) ──
vectors_cache = OUTPUT_DIR / "identity_vectors"

if vectors_cache.exists() and (vectors_cache / "metadata.json").exists():
    print("Loading cached identity vectors...")
    identity_vectors = IdentityVectorSet.load(str(vectors_cache))
    print(f"  Loaded from {vectors_cache}")
else:
    print("Computing identity vectors from scratch...")
    with SlopClient(cfg) as client:
        info = client.get_server_info()
        print(f"  Connected: {info.gpu_name} | CUDA {info.cuda_version}")

        identity_vectors = compute_identity_vectors(
            client,
            identities=IDENTITIES,
            anchor=ANCHOR,
            model_id=MODEL,
        )
    identity_vectors.save(str(vectors_cache))

# Show vector magnitudes (larger = more distinct embedding direction)
print(f"\nIdentity vector magnitudes:")
for name, vec in identity_vectors.vectors.items():
    mag = np.linalg.norm(vec)
    print(f"  {name:>13}: ‖v‖ = {mag:.4f}")

print(f"\nEmbedding shape: {identity_vectors.embed_shape}")

# ── Encode theme and build conditioned embeddings ──
print(f"\nEncoding theme: \"{THEME}\"...")
with SlopClient(cfg) as client:
    theme_embed = client.get_prompt_embeds(THEME, model_id=MODEL)

conditioned_embeds = build_conditioned_embeds(
    theme_embed, identity_vectors,
    categories=list(IDENTITIES.keys()),
    scale=IDENTITY_SCALE,
)

print(f"\nConditioned embeddings ready:")
for cat, embed in conditioned_embeds.items():
    cos_sim = np.dot(embed.flatten(), theme_embed.flatten()) / (
        np.linalg.norm(embed.flatten()) * np.linalg.norm(theme_embed.flatten())
    )
    print(f"  {cat:>13}: cos(theme, conditioned) = {cos_sim:.6f}")

Computing identity vectors from scratch...
[REMOTE STDERR] Welcome to vast.ai. If authentication fails, try again after a few seconds, and double check your ssh key.
[REMOTE STDERR] Have fun!
[REMOTE STDERR] [SERVER] 2026-02-08 16:48:25,928 - INFO - SLOP Inference Server started. Waiting for input on stdin...
  Connected: NVIDIA GeForce RTX 3090 | CUDA 12.1
[REMOTE STDERR] [SERVER] 2026-02-08 16:48:26,889 - INFO - Encoding 6 text(s) via pipeline text encoder
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  7.99it/s]0:00,  7.80it/s]
Saved identity vectors to /Users/pkd/Desktop/slop/results/targeted_daily_life_in_jaffa/identity_vectors

Identity vector magnitudes:
           arab: ‖v‖ = 205.3668
    palestinian: ‖v‖ = 201.6974
         jewish: ‖v‖ = 196.1320
          jesus: ‖v‖ = 232.5266

Embedding shape: (1, 77, 768)

Encoding theme: "daily life in Jaffa"...
[REMOTE STDERR] Welcome to vast.ai. If authentication fails, try again after a few seconds, and double check 


---
# Part 2 — Image Evolution Analysis

## How Does an Existing Image *Change* Under Identity-Conditioned Denoising?

**Concept:** Take a real image (or a model-generated "neutral" image), encode it into latent space, and then run partial denoising (img2img) with different identity-conditioned prompts. By comparing the trajectories:

1. **Which pixels stay fixed** — structurally important regions the model preserves
2. **Which pixels move** — what the model "edits" when told to add identity
3. **How the image evolves globally** — direction and magnitude of the edit
4. **At what denoising strength** do colonial patterns emerge?

This is like watching the model "repaint" parts of a neutral scene to make it match its idea of what an identity looks like — revealing which visual elements it considers essential to each identity.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  IMAGE EVOLUTION CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════


REFERENCE_IMAGE_PATH = project_root / "data/historical/raw/damascus-gate.jpg"  # set to a path if you want to use your own image

# Denoising strengths to test: 0 = no change, 1 = full regeneration
# The interesting zone is 0.3-0.7 where structure is partially preserved
DENOISE_STRENGTHS = [0.2, 0.35, 0.5, 0.65, 0.8]

# Which identities to test in evolution (uses same identity vectors)
EVOLUTION_CATEGORIES = ["neutral", "arab", "palestinian"]

N_EVOLUTION_SEEDS = 20  # seeds per strength × identity combo

EVOLUTION_DIR = OUTPUT_DIR / "evolution"
EVOLUTION_DIR.mkdir(exist_ok=True)
print("Evolution config ready ✓")
print(f"  Categories: {EVOLUTION_CATEGORIES}")
print(f"  Strengths:  {DENOISE_STRENGTHS}")
print(f"  Seeds:      {N_EVOLUTION_SEEDS}")

Evolution config ready ✓
  Categories: ['neutral', 'arab', 'palestinian']
  Strengths:  [0.2, 0.35, 0.5, 0.65, 0.8]
  Seeds:      20


In [7]:
# ── Select or load reference image ──


reference_image = Path(REFERENCE_IMAGE_PATH).read_bytes()
reference_latent = None
print(f"Loaded reference from {REFERENCE_IMAGE_PATH}")


Loaded reference from /Users/pkd/Desktop/slop/data/historical/raw/damascus-gate.jpg


### Run Image-to-Image Evolution

For each denoising strength × identity prompt, run N seeds of img2img. The reference image is noised to the target strength, then denoised with the identity-conditioned prompt. This lets us see exactly what the model changes.

In [ ]:
# ── Generate img2img variations using embedding overrides ──
# NOTE: True img2img requires server support (init_image in InferenceRequest).
# For now we generate full trajectories with identity-conditioned embeddings at each
# "strength" — where strength controls the identity vector scale.
# This simulates the question: "how much identity do you need to inject before
# the model starts producing colonial imagery?"

evolution_results: dict[str, dict[float, list]] = defaultdict(lambda: defaultdict(list))
evolution_latents: dict[str, dict[float, list]] = defaultdict(lambda: defaultdict(list))

with SlopClient(cfg) as client:
    info = client.get_server_info()
    print(f"Connected: {info.gpu_name}")

    for strength in DENOISE_STRENGTHS:
        print(f"\n{'='*60}")
        print(f"  Identity scale = {strength}")
        print(f"{'='*60}")

        # Build embeddings at this scale
        scaled_embeds = build_conditioned_embeds(
            theme_embed, identity_vectors,
            categories=[c for c in EVOLUTION_CATEGORIES if c != "neutral"],
            scale=strength,
        )

        for cat in EVOLUTION_CATEGORIES:
            embed = scaled_embeds[cat]
            neg_embed = identity_vectors.negative_embed

            print(f"  [{cat}] α={strength} ... ", end="", flush=True)

            cat_results = []
            cat_lats = []

            for seed in range(N_EVOLUTION_SEEDS):
                result = client.generate_with_embeds(
                    prompt_embeds=embed,
                    negative_prompt_embeds=neg_embed,
                    num_steps=NUM_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    seed=seed,
                    model_id=MODEL,
                )

                if result.latents is not None:
                    cat_lats.append(result.latents)
                cat_results.append(result)

            evolution_results[cat][strength] = cat_results
            evolution_latents[cat][strength] = cat_lats
            print(f"✓ {len(cat_lats)} trajectories")

            # Save
            s_dir = EVOLUTION_DIR / f"scale_{strength:.2f}" / cat
            s_dir.mkdir(parents=True, exist_ok=True)
            for i, lat in enumerate(cat_lats):
                np.save(s_dir / f"latents_{i:03d}.npy", lat)

print("\n✓ Evolution generation complete")

[REMOTE STDERR] Welcome to vast.ai. If authentication fails, try again after a few seconds, and double check your ssh key.
[REMOTE STDERR] Have fun!
[REMOTE STDERR] [SERVER] 2026-02-08 16:55:33,233 - INFO - SLOP Inference Server started. Waiting for input on stdin...
Connected: NVIDIA GeForce RTX 3090

  Identity scale = 0.2
  [neutral] α=0.2 ... [REMOTE STDERR] [SERVER] 2026-02-08 16:55:35,373 - INFO - Running inference: ...


### Analyze: What Moves vs What Stays?

For each denoising strength, compare the identity-conditioned result against the neutral result. The **spatial difference map** reveals which image regions the model considers identity-relevant.

In [ ]:
# ── Spatial stability analysis across identity vector scales ──

non_neutral_evo = [c for c in EVOLUTION_CATEGORIES if c != "neutral"]
fig_rows = len(non_neutral_evo)
fig_cols = len(DENOISE_STRENGTHS)

fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(5 * fig_cols, 5 * fig_rows))
if fig_rows == 1:
    axes = axes[np.newaxis, :]

for row, cat in enumerate(non_neutral_evo):
    for col, strength in enumerate(DENOISE_STRENGTHS):
        ax = axes[row, col]

        cat_lats = evolution_latents[cat][strength]
        neut_lats = evolution_latents["neutral"][strength]

        if not cat_lats or not neut_lats:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            continue

        # Compute spatial heatmap: where does identity change the image?
        bsr = batch_spatial_heatmap(cat_lats, neut_lats, step=-1)

        im = ax.imshow(bsr.snr, cmap="RdYlGn", interpolation="bilinear",
                       vmin=0, vmax=5)
        ax.set_title(f"α={strength}  SNR\nmed={np.median(bsr.snr):.1f}", fontsize=10)
        ax.axis("off")

        if col == 0:
            ax.set_ylabel(cat, fontsize=12, fontweight="bold", rotation=0,
                         labelpad=50, va="center")

plt.suptitle(
    f"Spatial SNR: What the model changes at each identity vector scale\n"
    f"\"{THEME}\" — green = reliably modified, red = preserved",
    fontsize=14,
)
plt.tight_layout()
plt.savefig(EVOLUTION_DIR / "stability_map.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Divergence as a function of identity vector scale ──
# At what scale does colonial imagery start to appear?

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Final divergence vs scale
ax = axes[0]
for cat in non_neutral_evo:
    scale_vals = []
    divs = []
    divs_sem = []

    for strength in DENOISE_STRENGTHS:
        cat_lats = evolution_latents[cat][strength]
        neut_lats = evolution_latents["neutral"][strength]
        if len(cat_lats) >= 2 and len(neut_lats) >= 2:
            bdr = batch_divergence(cat_lats, neut_lats, n_perms=200, ci=0.95)
            scale_vals.append(strength)
            divs.append(bdr.mean[-1])
            divs_sem.append(bdr.sem[-1])

    c = COLORS.get(cat, "gray")
    ax.errorbar(scale_vals, divs, yerr=[1.96 * s for s in divs_sem],
                marker="o", linewidth=2, color=c, capsize=5, label=cat)

ax.set_xlabel("Identity vector scale (α)", fontsize=12)
ax.set_ylabel("Final L2 divergence", fontsize=12)
ax.set_title("Divergence vs identity vector scale\n(when does colonial imagery appear?)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 2: Spatial SNR median vs scale
ax2 = axes[1]
for cat in non_neutral_evo:
    scale_vals = []
    snrs = []

    for strength in DENOISE_STRENGTHS:
        cat_lats = evolution_latents[cat][strength]
        neut_lats = evolution_latents["neutral"][strength]
        if len(cat_lats) >= 2 and len(neut_lats) >= 2:
            bsr = batch_spatial_heatmap(cat_lats, neut_lats, step=-1)
            scale_vals.append(strength)
            snrs.append(np.median(bsr.snr))

    c = COLORS.get(cat, "gray")
    ax2.plot(scale_vals, snrs, "o-", linewidth=2, color=c, label=cat)

ax2.axhline(2.0, color="gray", linestyle="--", alpha=0.5, label="SNR=2 threshold")
ax2.set_xlabel("Identity vector scale (α)", fontsize=12)
ax2.set_ylabel("Median spatial SNR", fontsize=12)
ax2.set_title("Spatial reliability vs identity scale\n(above 2 = reliable colonial pattern)", fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle(f"Identity Vector Scale Analysis — \"{THEME}\"", fontsize=15)
plt.tight_layout()
plt.savefig(EVOLUTION_DIR / "scale_vs_divergence.png", dpi=150)
plt.show()

In [ ]:
# ── Per-pixel movement map: overlay on reference image ──
# Show which parts of the image the model "wants to change" for each identity

if reference_image:
    ref_img = Image.open(BytesIO(reference_image)).resize((64, 64))  # match latent grid
    ref_array = np.array(ref_img).astype(float) / 255.0

    # Pick the middle identity scale for the overlay
    mid_strength = DENOISE_STRENGTHS[len(DENOISE_STRENGTHS) // 2]

    fig, axes = plt.subplots(1, len(non_neutral_evo) + 1, figsize=(6 * (len(non_neutral_evo) + 1), 6))

    # Reference
    axes[0].imshow(Image.open(BytesIO(reference_image)))
    axes[0].set_title("Reference (neutral)", fontsize=12)
    axes[0].axis("off")

    for i, cat in enumerate(non_neutral_evo):
        ax = axes[i + 1]
        cat_lats = evolution_latents[cat][mid_strength]
        neut_lats = evolution_latents["neutral"][mid_strength]

        if len(cat_lats) >= 2 and len(neut_lats) >= 2:
            bsr = batch_spatial_heatmap(cat_lats, neut_lats, step=-1)

            # Overlay: use SNR as a mask over the reference image
            snr_norm = np.clip(bsr.snr / max(bsr.snr.max(), 1), 0, 1)

            # Create RGBA overlay: colored where model changes things
            overlay = np.zeros((*snr_norm.shape, 4))
            c_rgb = tuple(int(COLORS.get(cat, "#FF0000").lstrip("#")[i:i+2], 16) / 255 for i in (0, 2, 4))
            overlay[:, :, 0] = c_rgb[0]
            overlay[:, :, 1] = c_rgb[1]
            overlay[:, :, 2] = c_rgb[2]
            overlay[:, :, 3] = snr_norm * 0.6  # alpha proportional to SNR

            ax.imshow(ref_array)
            ax.imshow(overlay, interpolation="bilinear")
            ax.set_title(f"{cat} (α={mid_strength})\nColored = model edits", fontsize=12)
        else:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    plt.suptitle(
        f"What the model changes for each identity — \"{THEME}\"\n"
        f"Overlay = spatial SNR at scale {mid_strength} (color intensity ∝ edit reliability)",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(EVOLUTION_DIR / "edit_overlay.png", dpi=150, bbox_inches="tight")
    plt.show()

### Evolution Filmstrip — Watch Identity Vector Strength Increase

For a single seed, show how the neutral image progressively changes at increasing identity vector scales. This is like watching the model "apply" colonial visual grammar with increasing intensity.

In [ ]:
# Show images at each identity scale for one seed
FILMSTRIP_SEED = 0

n_scales = len(DENOISE_STRENGTHS)
n_ids = len(EVOLUTION_CATEGORIES)

fig, axes = plt.subplots(n_ids, n_scales + 1, figsize=(3 * (n_scales + 1), 3.5 * n_ids))

for row, cat in enumerate(EVOLUTION_CATEGORIES):
    # Column 0: reference
    ax = axes[row, 0]
    if reference_image:
        ax.imshow(Image.open(BytesIO(reference_image)))
    ax.set_title("Reference" if row == 0 else "", fontsize=10)
    if row == 0:
        ax.set_title("Reference\n(neutral)", fontsize=10)
    ax.set_ylabel(cat, fontsize=11, fontweight="bold", rotation=0, labelpad=50, va="center")
    ax.axis("off")

    # Columns 1..N: each identity vector scale
    for col, strength in enumerate(DENOISE_STRENGTHS):
        ax = axes[row, col + 1]
        results = evolution_results.get(cat, {}).get(strength, [])
        if FILMSTRIP_SEED < len(results) and results[FILMSTRIP_SEED].image:
            try:
                img = Image.open(BytesIO(results[FILMSTRIP_SEED].image))
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "error", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)
        if row == 0:
            ax.set_title(f"α={strength}", fontsize=10)
        ax.axis("off")

plt.suptitle(
    f"Identity Vector Scale Filmstrip — \"{THEME}\" (seed {FILMSTRIP_SEED})\n"
    f"Left to right: increasing identity vector strength → more identity-specific features",
    fontsize=14, y=1.02,
)
plt.tight_layout()
plt.savefig(EVOLUTION_DIR / "filmstrip.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Methodology Notes

### Identity Vector Method
- **Embedding-space control:** Instead of varying the text prompt (which introduces tokenization artifacts and contextual interaction between theme and identity words), we compute a fixed direction vector in the text encoder's output space.
- **The identity vector:** $\vec{v}_\text{id} = \text{enc}(\text{"Arab person"}) - \text{enc}(\text{"a person"})$ captures the *pure* direction of the identity concept, independent of any scene.
- **Conditioned generation:** $\vec{e}_\text{cond} = \text{enc}(\text{theme}) + \alpha \cdot \vec{v}_\text{id}$ — the UNet receives the modified embedding directly, bypassing the text encoder entirely for the experimental variable.
- **Perfect control:** The ONLY difference between categories is the identity vector. Same theme embedding, same negative embedding, same seeds, same model weights.

### Targeted Analysis (Part 1)
- **Single theme:** By fixing the scene and only varying the identity vector, we control for all confounds.
- **N=100:** With 100 seeds, SEM ≈ σ/10. A systematic difference of even 10% of the per-seed standard deviation becomes statistically detectable.
- **Multiple identities:** Including "jewish" and "jesus" as controls helps distinguish colonial-specific from Middle-Eastern-general patterns.

### Identity Vector Scale Sweep (Part 2)
- **Continuous control:** By varying α from 0 to 2, we can find the *threshold* at which colonial visual patterns emerge. This is like a dose-response curve for colonial imagery.
- **Spatial stability maps:** Regions with high SNR across seeds are where the model *consistently* makes identity-specific edits — these are the colonial visual signatures.
- **The key question:** At what identity vector strength does the model start adding gates, walls, dust, specific architectural elements? The scale sweep answers this directly.

### Suggested Themes to Try
- `"daily life in Jaffa"` — local Mediterranean identity
- `"a beach scene"` — neutral setting, identity should be irrelevant
- `"a family having dinner"` — domestic setting
- `"a market scene"` — commercial/social setting
- `"a school classroom"` — institutional setting
- `"a wedding celebration"` — cultural event
- `"a city street at night"` — urban setting